In [24]:
import pandas as pd
import networkx as nx

given_neurons = pd.read_csv("given_neurons.csv")
edge_list = ( # edges are separated by neuropil, but we only care about total synapse count. group by edges and sum synapse counts
    pd.read_csv('connections_princeton.csv')
    .drop(columns=['nt_type'])
    .groupby(['pre_root_id', 'post_root_id'], as_index=False)['syn_count']
    .sum()
)

G = nx.from_pandas_edgelist(edge_list, source='pre_root_id', target='post_root_id', create_using=nx.DiGraph)

# Part 1: ORN -> OviDN

In [ ]:
ORN = given_neurons[given_neurons['group'] == 'ORN']['root_id']
OVIDN = given_neurons[given_neurons['group'] == 'OviDN']['root_id']
path_edges = []

iters = len(ORN) * len(OVIDN)
counter = 1
for orn in ORN:
    for ovidn in OVIDN:
        print(f" ({counter}/{iters}) finding pathways between {orn} -> {ovidn}...", end="")
        paths: Generator[list, None, None] = nx.algorithms.all_shortest_paths(G, orn, ovidn)
        print("\r", end="")
        for path in paths:
            # if len(path) > 4: continue # only 3 hops/4 neurons or less are considered as per the original paper's methods
            path_edges.extend(zip(path, path[1:])) # zip(path, path[1:]) is a neat shorthand of generating edges from the list of nodes in the path
            # INTERNEURONS.update(set(path[1:-1])) # each path starts with Ir94e and ends with OviDN, so the interneurons are the nodes in between
        counter += 1

path_edges_df = pd.DataFrame(path_edges, columns=["pre_root_id", "post_root_id"])
print(f"unique path edges discovered: {len(path_edges_df.drop_duplicates())}")

 (595/54108) finding pathways between 720575941436366624 -> 720575941417243164...

KeyboardInterrupt: 